# TP 2 — Construire (et dépasser) son propre système de traduction statistique

**Fil rouge de ce TP :** en 1993, IBM Model 1 + un décodeur simple, c'était l'état de l'art. Vous allez le reconstruire vous-même, sur votre vrai corpus, puis mesurer exactement l'écart qui nous sépare de 2024/2025 (NLLB, LLM). Le nombre que vous allez obtenir à la fin est un vrai résultat expérimental — personne ne connaît sa valeur à l'avance pour votre corpus.

**Vous allez :**
1. aligner vos phrases (vous l'avez déjà fait — on réutilise l'outillage) ;
2. implémenter vous-même le E-step et le M-step de l'algorithme EM pour IBM Model 1 ;
3. en tirer un **lexique bilingue automatique** — un sous-produit réellement utilisé pour les langues peu dotées ;
4. construire un traducteur complet avec le canal bruité (modèle de traduction + modèle de langue + décodeur) ;
5. **évaluer** vos traductions (BLEU/sacreBLEU, COMET si disponible) et les comparer à NLLB-200 ou un LLM ;
6. discuter, avec vos propres exemples, de ce qui casse — et pourquoi.

Les cellules marquées **`# TODO`** sont à compléter. Le reste est fourni pour que vous puissiez vous concentrer sur l'algorithme, pas sur la plomberie.

## 0. Installation

In [ ]:
!pip -q install sentence-transformers sacrebleu unbabel-comet nltk datasets
import re, math, random
from collections import defaultdict, Counter
import numpy as np
random.seed(0)

## 1. Charger votre corpus

Déposez vos fichiers `src.txt` (langue source) et `tgt.txt` (langue cible, une unité par ligne). Le code cherche ces fichiers ; à défaut, il utilise un petit corpus-jouet allemand → français (inventé, pour tester que le pipeline tourne — **remplacez-le par votre corpus dès que possible**).

In [ ]:
import os

toy_src = [
    "das Haus ist groß", "das Buch ist alt", "die Katze schläft", "der Hund läuft",
    "das Kind liest das Buch", "die Frau kauft das Haus", "der Mann liest die Zeitung",
    "das Mädchen sieht die Katze", "der Hund sieht das Kind", "die Frau liest das Buch",
]
toy_tgt = [
    "la maison est grande", "le livre est vieux", "le chat dort", "le chien court",
    "l'enfant lit le livre", "la femme achète la maison", "l'homme lit le journal",
    "la fille voit le chat", "le chien voit l'enfant", "la femme lit le livre",
]

def load_lines(path, toy):
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return [l.strip() for l in f if l.strip()]
    print(f"[info] {path} introuvable — corpus-jouet utilisé.")
    return toy

src_sents = load_lines("src.txt", toy_src)
tgt_sents = load_lines("tgt.txt", toy_tgt)
assert len(src_sents) == len(tgt_sents), "src et tgt doivent avoir le même nombre de lignes : alignez-les d'abord (section 2) !"
print(len(src_sents), "paires de phrases chargées.")

## 2. Alignement des phrases — à vous de jouer

Si vos deux fichiers ne sont **pas déjà alignés ligne à ligne** (segmentation littéraire, nombre de phrases différent...), c'est ici qu'on le corrige — exactement comme dans le TP précédent. Réutilisez la méthode qui avait le mieux marché sur votre corpus (`bertalign`, ou votre réaligneur par embeddings fait main).

In [ ]:
# TODO : alignez vos fichiers
#
# Réutilisez votre fonction du TP précédent
#
# Puis reconstruisez src_sents / tgt_sents à partir de `pairs`.


In [ ]:
# Vous pouvez utiliser les textes de "Petit Prince":

from datasets import load_dataset # explorez la librairie "datasets" de HugginFace

ds = load_dataset("averoo/low_resource_parallel_corpora")

## 3. Tokenisation et vocabulaire

In [ ]:
## attention à la ponctuation, adaptez la fonction suivante:

def tokenize(s):
    return s.lower().split()

src_tok = [tokenize(s) for s in src_sents]
tgt_tok = [tokenize(s) for s in tgt_sents]

src_vocab = sorted(set(w for s in src_tok for w in s))
tgt_vocab = sorted(set(w for s in tgt_tok for w in s)) + ["<NULL>"]

print(f"{len(src_vocab)} mots source, {len(tgt_vocab)} mots cible (NULL compris).")
print(src_tok[0], "<->", tgt_tok[0])

## 4. IBM Model 1 : construire le modèle de traduction

On va y aller **une seule idée à la fois**. Toute la mécanique (les boucles, l'accumulation sur le corpus) est déjà écrite pour vous : vous allez compléter **une seule fonction**, qui répond à une question très concrète.

Pour un mot source $f$, et les mots cibles candidats d'une phrase (ex. : la, maison, NULL), comment répartir « 1 unité de certitude » entre ces candidats, sachant les $t(f\mid e)$ actuels ?

### Un exemple numérique avant le code

Reprenons l'exemple de la Leçon 4. Phrase source : *das Haus*. Mots cibles candidats : *la*, *maison*. Avec $t$ initialisé uniformément à $\frac{1}{3}$ partout :

$$\text{dénominateur} = t(\text{das}\mid \text{la}) + t(\text{das}\mid \text{maison}) = \tfrac{1}{3} + \tfrac{1}{3} = \tfrac{2}{3}$$

$$P(a = \text{la} \mid \text{das}) = \frac{t(\text{das}\mid\text{la})}{\text{dénominateur}} = \frac{1/3}{2/3} = 0{,}5 \qquad P(a = \text{maison} \mid \text{das}) = 0{,}5$$

C'est tout ce que votre fonction doit calculer : **une division**, répétée pour chaque candidat.

In [ ]:
def posteriors_for_word(f, e_candidats, t):
    """
    f           : un mot source (ex. 'das')
    e_candidats : la liste des mots cible de la phrase, NULL compris (ex. ['<NULL>', 'la', 'maison'])
    t           : le dictionnaire courant des t(f|e)

    Retourne un dictionnaire {e: probabilité} qui doit sommer à 1.
    """
    denominateur = sum(t[(f, e)] for e in e_candidats)  # <- fourni, c'est la somme du bas de la formule

    probas = {}
    for e in e_candidats:
        probas[e] = 0  # TODO : remplacez le 0 par la vraie formule
        # Indice : c'est exactement le calcul fait à la main juste au-dessus.

    return probas

In [ ]:
# Testez votre fonction ici avant de continuer : avec t uniforme, vous devez obtenir 0,5 / 0,5.
t_test = defaultdict(lambda: 1.0 / 3)
resultat = posteriors_for_word("das", ["la", "maison"], t_test)
print(resultat)
assert abs(sum(resultat.values()) - 1.0) < 1e-6, "Les probabilités doivent sommer à 1."
assert abs(resultat["la"] - 0.5) < 1e-6, "Avec t uniforme, on doit retrouver 0,5 / 0,5."
print("Test réussi !")

**Si le test passe, vous avez fini la partie difficile.** Le reste — répéter ce calcul sur tout le corpus, puis ré-estimer $t(f\mid e)$ — est entièrement fourni ci-dessous ; lisez-le, il ne fait qu'appeler la fonction que vous venez d'écrire.

In [ ]:
def e_step(src_tok, tgt_tok, t):
    """Parcourt tout le corpus et accumule les comptes fractionnaires, en
    appelant posteriors_for_word() pour chaque mot source rencontré."""
    count = defaultdict(float)
    total_e = defaultdict(float)
    for f_sent, e_sent_raw in zip(src_tok, tgt_tok):
        e_sent = ["<NULL>"] + e_sent_raw
        for f in f_sent:
            probas = posteriors_for_word(f, e_sent, t)
            for e, p in probas.items():
                count[(f, e)] += p
                total_e[e] += p
    return count, total_e

def m_step(count, total_e):
    """Ré-estime t(f|e) = compte(f,e) / total(e). Une paire jamais vue ensemble reste à 0."""
    new_t = defaultdict(float)
    for (f, e), c in count.items():
        new_t[(f, e)] = c / total_e[e]
    return new_t

## 5. Entraînement : faites tourner EM (Expectation-Optimisation)

In [ ]:
# Initialisation : t(f|e) uniforme sur tout le vocabulaire source, pour un vrai entraînement
t = defaultdict(lambda: 1.0 / len(src_vocab))

N_ITER = 10
for it in range(N_ITER):
    count, total_e = e_step(src_tok, tgt_tok, t)
    t = m_step(count, total_e)

# petit test de cohérence : la somme des t(f|e) sur f, pour un e fixé, doit faire ~1
s = sum(t[(f, tgt_vocab[0])] for f in src_vocab)
print(f"Somme de contrôle pour e={tgt_vocab[0]!r} (doit être proche de 1) : {s:.3f}")

## 6. Sous-produit utile : extraire un lexique bilingue

Pour chaque mot cible, quel mot source lui est le plus probablement associé ? C'est exactement ce que font les dictionnaires bilingues construits automatiquement pour les langues peu dotées.

In [ ]:
def top_translation(e, k=3):
    scores = [(t[(f, e)], f) for f in src_vocab]
    scores.sort(reverse=True)
    return scores[:k]

for e in tgt_vocab:
    if e == "<NULL>":
        continue
    best = top_translation(e)
    print(f"{e:12s} -> " + ", ".join(f"{f} ({p:.2f})" for p, f in best))

**Question :** certains mots cible pointent-ils de façon convaincante vers NULL (aucun mot source ne leur correspond vraiment) ? Que représentent ces cas concrètement ?

## 7. Modèle de langue (fourni)

Un bigramme avec lissage de Laplace, entraîné sur le **monolingue cible** uniquement — exactement l'esprit du canal bruité : $P(e)$ ne nécessite aucune donnée parallèle.

In [ ]:
class BigramLM:
    def __init__(self, sentences):
        self.unigrams = Counter()
        self.bigrams = Counter()
        self.vocab = set()
        for s in sentences:
            toks = ["<s>"] + tokenize(s) + ["</s>"]
            self.vocab.update(toks)
            for w in toks:
                self.unigrams[w] += 1
            for w1, w2 in zip(toks, toks[1:]):
                self.bigrams[(w1, w2)] += 1
        self.V = len(self.vocab)

    def logprob(self, sentence_tokens):
        toks = ["<s>"] + sentence_tokens + ["</s>"]
        lp = 0.0
        for w1, w2 in zip(toks, toks[1:]):
            num = self.bigrams[(w1, w2)] + 1
            den = self.unigrams[w1] + self.V
            lp += math.log(num / den)
        return lp

lm = BigramLM(tgt_sents)
print("logP('le chat dort') =", lm.logprob(tokenize("le chat dort")))

## 8. Le décodeur : combiner $P(f \mid e)$ et $P(e)$

On simplifie radicalement la recherche ($\arg\max_e$ sur **toutes** les phrases possibles est impraticable à la main) : pour chaque mot source, on ne garde que ses $k$ meilleures traductions candidates, on génère des combinaisons, et on les score. Toute la mécanique de recherche est fournie — vous n'écrivez que **le score d'une seule combinaison**, qui est une simple addition.

In [ ]:
import itertools, random

def candidate_words(f, k=3):
    """Les k meilleures traductions candidates d'un mot source, d'après t(f|e). Fourni."""
    scores = [(t[(f, e)], e) for e in tgt_vocab if e != "<NULL>"]
    scores.sort(reverse=True)
    return [e for _, e in scores[:k]]

def decode_greedy(f_sent_tokens):
    """Traduction mot-à-mot gloutonne : pour chaque mot source, prend sa meilleure traduction. Fourni."""
    return " ".join(candidate_words(f, k=1)[0] for f in f_sent_tokens)

### À vous : noter une combinaison candidate

Pour une phrase source $(f_1, \dots, f_m)$ et une combinaison candidate $(e_1, \dots, e_m)$ (une traduction candidate pour chaque mot, dans l'ordre), le score du canal bruité est :

$$\text{score} = \underbrace{\sum_{j=1}^{m} \log t(f_j \mid e_j)}_{\text{modèle de traduction}} \;+\; \underbrace{\log P(e_1, \dots, e_m)}_{\text{modèle de langue}}$$

Le second terme, c'est exactement `lm.logprob(...)` de la section 7.

In [ ]:
def score_combo(f_sent_tokens, combo):
    """
    f_sent_tokens : les mots source, ex. ['die', 'katze', 'läuft']
    combo         : une combinaison candidate de mots cible, même longueur, ex. ('le', 'chat', 'court')

    Retourne le score du canal bruité (un seul nombre, plus il est grand, mieux c'est).
    """
    score_traduction = 0  # TODO : additionnez log(t(f_j | e_j)) pour chaque paire (f_j, e_j)
    # Indice : zip(f_sent_tokens, combo) donne les paires (f_j, e_j) dans l'ordre.
    # Indice 2 : utilisez math.log(t[(f, e)] + 1e-12) -- le +1e-12 évite log(0).

    score_langue = lm.logprob(list(combo))  # <- fourni

    return score_traduction + score_langue

In [ ]:
def score_combo(f_sent_tokens, combo):
    """
    f_sent_tokens : les mots source, ex. ['die', 'katze', 'läuft']
    combo         : une combinaison candidate de mots cible, même longueur, ex. ('le', 'chat', 'court')

    Retourne le score du canal bruité (un seul nombre, plus il est grand, mieux c'est).
    """
    score_traduction = 0  # TODO : additionnez log(t(f_j | e_j)) pour chaque paire (f_j, e_j)
    # Indice : zip(f_sent_tokens, combo) donne les paires (f_j, e_j) dans l'ordre.
    # Indice 2 : utilisez math.log(t[(f, e)] + 1e-12) -- le +1e-12 évite log(0).

    score_langue = lm.logprob(list(combo))  # <- fourni

    return score_traduction + score_langue

Le reste — générer les combinaisons candidates et garder la meilleure — est fourni : il ne fait qu'appeler `score_combo()` autant de fois que nécessaire.

In [ ]:
def decode_noisy_channel(f_sent_tokens, k=3, n_samples=200):
    """Génère des combinaisons candidates (produit cartésien si petit, sinon tirage aléatoire),
    et renvoie celle qui maximise score_combo(). Fourni."""
    cand_lists = [candidate_words(f, k=k) for f in f_sent_tokens]

    total_combos = 1
    for c in cand_lists:
        total_combos *= len(c)

    if total_combos <= n_samples:
        combos = itertools.product(*cand_lists)
    else:
        combos = (tuple(random.choice(c) for c in cand_lists) for _ in range(n_samples))

    best_score, best_combo = -1e18, None
    for combo in combos:
        s = score_combo(f_sent_tokens, combo)
        if s > best_score:
            best_score, best_combo = s, combo

    return " ".join(best_combo) if best_combo else ""

test_sent = tokenize("die Katze läuft")
print("Mot-à-mot :", decode_greedy(test_sent))
print("Canal bruité :", decode_noisy_channel(test_sent))

**Question :** sur quels types de phrases le décodeur « canal bruité » fait-il mieux que le simple mot-à-mot ? Sur quels types ne fait-il pas mieux — et pourquoi, d'après ce que vous savez des limites de Model 1 ?

## 9. Traduire un jeu de test

In [ ]:
# Séparez quelques phrases de test (idéalement non vues à l'entraînement, sur votre vrai corpus).
test_src = src_sents[-3:]
test_ref = tgt_sents[-3:]

system_hyps = [decode_noisy_channel(tokenize(s)) or decode_greedy(tokenize(s)) for s in test_src]

for s, h, r in zip(test_src, system_hyps, test_ref):
    print(f"SRC: {s}\nHYP: {h}\nREF: {r}\n")

## 10. Évaluation — votre système vs. l'état de l'art

On compare trois traducteurs sur le **même jeu de test** : votre IBM Model 1 + canal bruité, une traduction mot-à-mot (baseline minimale), et un modèle moderne (NLLB-200 ou un LLM via prompt).

In [ ]:
import sacrebleu

baseline_hyps = [decode_greedy(tokenize(s)) for s in test_src]

def show_bleu(name, hyps, refs):
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    print(f"{name:25s} sacreBLEU = {bleu.score:.1f}")
    return bleu.score

show_bleu("Mot-à-mot (baseline)", baseline_hyps, test_ref)
show_bleu("IBM Model 1 + canal bruité", system_hyps, test_ref)

### Comparaison avec un système moderne (fourni)

Adaptez le code de langue NLLB à votre paire de langues (voir la liste des codes FLORES-200). Si votre langue n'est pas couverte, essayez un LLM via prompt (section suivante) à la place.

In [ ]:
try:
    from transformers import pipeline
    # Adaptez les codes de langue NLLB à votre paire -- ex. "deu_Latn" -> "fra_Latn"
    translator = pipeline("translation", model="facebook/nllb-200-distilled-600M",
                           src_lang="deu_Latn", tgt_lang="fra_Latn")
    nllb_hyps = [translator(s)[0]["translation_text"] for s in test_src]
    for s, h in zip(test_src, nllb_hyps):
        print(f"SRC: {s}\nNLLB: {h}\n")
    show_bleu("NLLB-200 (600M)", nllb_hyps, test_ref)
except Exception as e:
    print("NLLB indisponible pour cette paire de langues :", e)
    print("-> utilisez un LLM via prompt (cellule suivante) comme point de comparaison.")

### Alternative / complément : COMET

sacreBLEU reste une mesure de surface. Si un modèle COMET couvre votre langue, ajoutez un score sémantique — surtout utile ici, où votre système statistique produira sans doute des phrases agrammaticales que BLEU seul ne décrit pas bien.

In [ ]:
try:
    from comet import download_model, load_from_checkpoint
    model_path = download_model("Unbabel/wmt22-comet-da")
    comet_model = load_from_checkpoint(model_path)
    data = [{"src": s, "mt": h, "ref": r} for s, h, r in zip(test_src, system_hyps, test_ref)]
    out = comet_model.predict(data, batch_size=8, gpus=0)
    print("COMET (IBM Model 1 + canal bruité) :", out.system_score)
except Exception as e:
    print("COMET indisponible :", e)

## 11. Bilan — questions pour votre compte-rendu

1. Comparez les trois scores sacreBLEU obtenus (mot-à-mot, IBM Model 1 + canal bruité, NLLB/LLM).
2. Donnez 3 exemples de phrases de votre corpus où votre système échoue, et reliez chaque échec à une hypothèse précise de IBM Model 1 (ordre des mots ? fertilité ? mot rare ?).
3. Le lexique bilingue extrait en section 6 est-il, lui, réellement utilisable ? Pour quel usage (pas forcément la traduction automatique complète) ?
4. Si vous deviez améliorer ce système sans passer au neuronal, quelle serait votre première piste (plus de données ? meilleur décodeur) ?
5. **Pour aller plus loin (optionnel) :** réentraînez avec plus d'itérations, ou sur un sous-ensemble plus grand de votre corpus — le score sacreBLEU de votre système progresse-t-il, et jusqu'où ?